In [ ]:
!pip install dash
!pip install pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 23.9 MB/s eta 0:00:00


In [40]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [41]:
import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/data/ecommerce_customer_churn_dataset.csv')
df.head()

,Age,Gender,Country,City,Membership_Years,Login_Frequency,Session_Duration_Avg,Pages_Per_Session,Cart_Abandonment_Rate,Wishlist_Items,...,Email_Open_Rate,Customer_Service_Calls,Product_Reviews_Written,Social_Media_Engagement_Score,Mobile_App_Usage,Payment_Method_Diversity,Lifetime_Value,Credit_Balance,Churned,Signup_Quarter
0,43.0,Male,France,Marseille,2.9,14.0,27.4,6.0,50.6,3.0,...,17.9,9.0,4.0,16.3,20.8,1.0,953.33,2278.0,0,Q1
1,36.0,Male,UK,Manchester,1.6,15.0,42.7,10.3,37.7,1.0,...,42.8,7.0,3.0,NaN,23.3,3.0,1067.47,3028.0,0,Q4
2,45.0,Female,Canada,Vancouver,2.9,10.0,24.8,1.6,70.9,1.0,...,0.0,4.0,1.0,NaN,8.8,NaN,1289.75,2317.0,0,Q4
3,56.0,Female,USA,New York,2.6,10.0,38.4,14.8,41.7,9.0,...,41.4,2.0,5.0,85.9,31.0,3.0,2340.92,2674.0,0,Q1
4,35.0,Male,India,Delhi,3.1,29.0,51.4,NaN,19.1,9.0,...,37.9,1.0,11.0,83.0,50.4,4.0,3041.29,5354.0,0,Q4


In [43]:
# Remove invalid ages
df = df[df["Age"].isna() | df["Age"].between(16, 100)]

# Fix values that should not be negative or above 100
df["Total_Purchases"] = df["Total_Purchases"].clip(lower=0)
df["Cart_Abandonment_Rate"] = df["Cart_Abandonment_Rate"].clip(0, 100)
df["Returns_Rate"] = df["Returns_Rate"].clip(0, 100)
df["Discount_Usage_Rate"] = df["Discount_Usage_Rate"].clip(0, 100)

# Fill missing values
df["Wishlist_Items"] = df["Wishlist_Items"].fillna(0)

# Fill missing numeric column values with median
numeric_cols = df.select_dtypes(include="number").columns
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())

# Convert selected columns to integers
int_cols = [
    "Age",
    "Customer_Service_Calls",
    "Product_Reviews_Written",
    "Wishlist_Items",
    "Payment_Method_Diversity",
    "Days_Since_Last_Purchase",
    "Login_Frequency"
]

df[int_cols] = df[int_cols].round().astype(int)

# Round remaining float columns to 2 decimal places
float_cols = df.select_dtypes(include="float").columns
df[float_cols] = df[float_cols].round(2)

# Clean text columns
text_cols = ["Gender", "Country", "City", "Signup_Quarter"]

for col in text_cols:
    df[col] = df[col].str.strip().str.title()

# Fix country abbreviations
df["Country"] = df["Country"].replace({
    "Uk": "UK",
    "Usa": "USA"
})

# Remove duplicate rows
df = df.drop_duplicates()

# Create useful dashboard columns
df["Churn_Status"] = df["Churned"].map({
    0: "Active",
    1: "Churned"
})

df["Payment_Mode"] = df["Payment_Method_Diversity"].map({
    1: "Cash",
    2: "Debit Card",
    3: "Credit Card",
    4: "PayPal",
    5: "Others"
})

df["Age_Group"] = pd.cut(
    df["Age"],
    bins=[16, 25, 35, 45, 55, 100],
    labels=["18-25", "26-35", "36-45", "46-55", "55 and above"]
)

df["Engagement_Tier"] = pd.cut(
    df["Login_Frequency"],
    bins=[-1, 4, 12, 24, 100],
    labels=["Low", "Medium", "High", "Very High"]
)

df["Value_Segment"] = pd.qcut(
    df["Lifetime_Value"],
    q=4,
    labels=["Bronze", "Silver", "Gold", "Platinum"]
)

# Save cleaned file
df.to_csv("churn_dashboard_clean.csv", index=True)

In [ ]:
from dash import Dash, html
from pyngrok import ngrok

ngrok.set_auth_token("3DMPEJ4Hmp7I7u1MvLQXIVEmYDU_6Zf6THWMbAopj2ZVh4Zdu")

app = Dash(__name__)

app.layout = html.Div([
    html.H1("Hello Dash on Google Colab"),
    html.P("Dash app running through ngrok")
])

public_url = ngrok.connect(8050)
print("Open this URL:", public_url)

app.run(port=8050)

Open this URL: NgrokTunnel: "https://chemicals-easily-seizing.ngrok-free.dev" -> "http://localhost:8050"
Dash is running on http://127.0.0.1:8050/



INFO:dash.dash:Dash is running on http://127.0.0.1:8050/



 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:8050
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [07/May/2026 14:00:41] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [07/May/2026 14:00:41] "GET /_dash-component-suites/dash/deps/polyfill@7.v4_1_0m1778161298.12.1.min.js HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [07/May/2026 14:00:41] "GET /_dash-component-suites/dash/deps/react@18.v4_1_0m1778161298.3.1.min.js HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [07/May/2026 14:00:41] "GET /_dash-component-suites/dash/deps/prop-types@15.v4_1_0m1778161298.8.1.min.js HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [07/May/2026 14:00:41] "GET /_dash-component-suites/dash/deps/react-dom@18.v4_1_0m1778161298.3.1.min.js HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [07/May/2026 14:00:42] "GET /_dash-component-suites/dash/dash-renderer/build/dash_renderer.v4_1_0m17781612